# Optional Gradio Front-End for SageMaker Serverless Endpoint

This notebook demonstrates how to build a simple **Gradio front-end UI** that invokes the SageMaker Serverless Endpoint created in the ITI113 SageMaker Pipeline tutorial.

Default example configuration:

| Item | Value |
|---|---|
| Team | `team40` |
| Student/Profile | `s4001` |
| Region | `ap-southeast-1` |
| Endpoint name | `iti113-team40-heart-disease` |

> **Important for students:** Change `TEAM_ID`, `STUDENT_ID`, and `ENDPOINT_NAME` to your own team/user/endpoint before running the notebook.

This Gradio app does **not train a model**. It only collects user input, formats the input as a CSV row, sends it to the deployed SageMaker Serverless Endpoint, and displays the prediction response.

## 1. Install Required Package

Run this once in SageMaker Studio. If Gradio is already installed, this cell will complete quickly.

In [ ]:
!pip install -q "starlette<1" "fastapi<1" --upgrade
!pip install gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.52.1 which is incompatible.


## 2. Configure Team, User, Region, and Check for Endpoint

For the class demo, the default endpoint is the Team40 endpoint:

```text
iti113-team40-heart-disease
```
Students must replace these values with their own team details.

This code searches for serverless endpoints created by own team

In [ ]:
import boto3
import json
import gradio as gr
from datetime import datetime

REGION = "ap-southeast-1"

# -------------------------------------------------------------------
# TODO: Students should change these values for their own team.
# Example:
# TEAM_ID = "team01"
# STUDENT_ID = "s101"
# ENDPOINT_NAME = "iti113-team01-heart-disease"
# -------------------------------------------------------------------
TEAM_ID = "team40"
STUDENT_ID = "s4001"
ENDPOINT_NAME = "iti113-team40-heart-disease"

sts = boto3.client("sts", region_name=REGION)

print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)
print("AWS identity:", sts.get_caller_identity()["Arn"])

sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

response = sm.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)

team_endpoints = []

for ep in response["Endpoints"]:
    endpoint_name = ep["EndpointName"]

    # Only show endpoints that follow the team naming convention
    if TEAM_ID in endpoint_name.lower():
        team_endpoints.append(ep)

print(f"\nEndpoints found for {TEAM_ID}: {len(team_endpoints)}\n")

for ep in team_endpoints:
    print("Endpoint name:", ep["EndpointName"])
    print("Status:", ep["EndpointStatus"])
    print("Creation time:", ep["CreationTime"])
    print("Last modified:", ep["LastModifiedTime"])
    print("-" * 80)

Region: ap-southeast-1
Team ID: team40
Student ID: s4001
AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team40/SageMaker

Endpoints found for team40: 1

Endpoint name: iti113-team40-heart-disease
Status: InService
Creation time: 2026-07-22 15:48:47.085000+00:00
Last modified: 2026-07-22 15:51:25.711000+00:00
--------------------------------------------------------------------------------


## 3. Confirm That the Serverless Endpoint(s) Is Active

The endpoint must show `InService` before it can be invoked.

In [35]:
try:
    for ep in team_endpoints:
        ENDPOINT_NAME = ep["EndpointName"]
        endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print("\nEndpoint name:", endpoint_desc["EndpointName"])
        print("Status:", endpoint_desc["EndpointStatus"])
        print("Creation time:", endpoint_desc["CreationTime"])
        print("Last modified:", endpoint_desc["LastModifiedTime"])
except Exception as e:
    print("Unable to describe endpoint.")
    print("Check that ENDPOINT_NAME is correct and that your role has permission to access it.")
    print("Error:", e)


Endpoint name: iti113-team40-heart-disease
Status: InService
Creation time: 2026-07-22 15:48:47.085000+00:00
Last modified: 2026-07-22 15:51:25.711000+00:00


## 4. Inspect the Endpoint Model Artifact (Optional)

This cell shows which SageMaker model and endpoint configuration are behind the endpoint. This helps students connect the deployed endpoint back to the saved model artifact produced by Notebook 03.

In [36]:
for ep in team_endpoints:
    ENDPOINT_NAME = ep["EndpointName"]

    print("=" * 100)
    print("Endpoint name:", ENDPOINT_NAME)

    try:
        # 1. Describe endpoint
        endpoint_desc = sm.describe_endpoint(
            EndpointName=ENDPOINT_NAME
        )

        endpoint_config_name = endpoint_desc["EndpointConfigName"]

        print("Endpoint status:", endpoint_desc["EndpointStatus"])
        print("Endpoint config:", endpoint_config_name)

        # 2. Describe endpoint config
        endpoint_config = sm.describe_endpoint_config(
            EndpointConfigName=endpoint_config_name
        )

        production_variants = endpoint_config.get("ProductionVariants", [])

        if not production_variants:
            print("No production variants found.")
            continue

        # 3. Loop through each production variant
        for variant in production_variants:
            print("-" * 80)

            variant_name = variant.get("VariantName")
            model_name = variant.get("ModelName")

            print("Variant name:", variant_name)
            print("Model name:", model_name)

            # Check whether endpoint is serverless or real-time instance
            if "ServerlessConfig" in variant:
                serverless_config = variant["ServerlessConfig"]
                print("Endpoint type: Serverless")
                print("Memory size:", serverless_config.get("MemorySizeInMB"), "MB")
                print("Max concurrency:", serverless_config.get("MaxConcurrency"))
            else:
                print("Endpoint type: Real-time instance")
                print("Instance type:", variant.get("InstanceType"))
                print("Initial instance count:", variant.get("InitialInstanceCount"))

            if not model_name:
                print("No model name found for this variant.")
                continue

            # 4. Describe model
            model_desc = sm.describe_model(
                ModelName=model_name
            )

            # 5. Handle PrimaryContainer or Containers
            if "PrimaryContainer" in model_desc:
                container = model_desc["PrimaryContainer"]

                print("Container format: PrimaryContainer")
                print("Primary container image:", container.get("Image", "Not shown"))
                print("Model artifact S3 URI:", container.get("ModelDataUrl", "Not shown"))

            elif "Containers" in model_desc:
                for i, container in enumerate(containers, start=1):
                    print(f"Container {i} image:", container.get("Image", "Not shown"))
                    print(f"Container {i} model artifact S3 URI:", container.get("ModelDataUrl", "Not shown"))
                
                    if "ModelPackageName" in container:
                        model_package_arn = container["ModelPackageName"]
                        print(f"Container {i} model package ARN:", model_package_arn)
                
                        try:
                            package_desc = sm.describe_model_package(
                                ModelPackageName=model_package_arn
                            )
                
                            inference_spec = package_desc.get("InferenceSpecification", {})
                            package_containers = inference_spec.get("Containers", [])
                
                            print("Model package approval status:", package_desc.get("ModelApprovalStatus", "Not shown"))
                
                            for j, pkg_container in enumerate(package_containers, start=1):
                                print(f"Package container {j} image:", pkg_container.get("Image", "Not shown"))
                                print(f"Package container {j} model artifact S3 URI:", pkg_container.get("ModelDataUrl", "Not shown"))
                
                        except Exception as e:
                            print("Unable to inspect model package details.")
                            print("Error:", e)
    except Exception as e:
        print("Unable to inspect this endpoint's model details.")
        print("Error:", e)

print("=" * 100)
print("Endpoint inspection completed.")

Endpoint name: iti113-team40-heart-disease
Endpoint status: InService
Endpoint config: iti113-team40-heart-disease
--------------------------------------------------------------------------------
Variant name: AllTraffic
Model name: team40-HeartDisease-2026-07-22-15-48-44-927
Endpoint type: Serverless
Memory size: 2048 MB
Max concurrency: 5


Container 1 image: Not shown
Container 1 model artifact S3 URI: Not shown
Container 1 model package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team40-HeartDisease/1
Model package approval status: Approved
Package container 1 image: 121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3
Package container 1 model artifact S3 URI: s3://sagemaker-ap-southeast-1-044528205969/sagemaker-scikit-learn-2026-07-11-15-35-47-349/pipelines-vonkbsbsyriz-RegisterModel-Repack-d1n8PIvKc9/output/model.tar.gz
Endpoint inspection completed.


## Inspecting the Deployed Endpoint

The endpoint is connected to a SageMaker Model, and the model may point to a Model Registry package instead of directly exposing the container image and model artifact.

For this Team40 example:

- Endpoint: `iti113-team40-heart-disease`
- Endpoint type: Serverless
- Model package: `team40-HeartDisease/1`
- Model package status: `Approved`
- Container image: SageMaker Scikit-learn container
- Model artifact: `model.tar.gz` in S3

This confirms that the endpoint is not just created, but is connected to an approved model package and a saved model artifact.

For Gradio UI, these three values are needed:

- REGION = "ap-southeast-1"
- TEAM_ID = "team40"          # change to your own team
- ENDPOINT_NAME = "iti113-team40-heart-disease"  # change to your own endpoint
 
### Check the endpoint invocation used previously from: 
#### 03 sagemaker_pipeline_mlflow_app_with_team_tag_check.ipynb
listed below (copied from notebook):

## 5. Test Endpoint Invocation with boto3

Before building the Gradio UI, test the endpoint directly using `boto3`.

The updated endpoint for this tutorial expects input in **JSON dictionary format**, not CSV. Each input field is sent using its feature name.

After updating Notebook 03, the deployed model bundle includes the trained model and the required preprocessing objects/metadata. This means the endpoint can now accept the conventional 13 Heart Disease input fields:

```text
age, sex, cp, trestbps, chol, fbs, restecg,
thalach, exang, oldpeak, slope, ca, thal
```

The endpoint `inference.py` will then perform the required preprocessing internally, including:

```text
age_group creation
high_risk_count creation
numeric scaling
feature ordering
prediction
```

This is different from the earlier endpoint version, where the request needed to include processed fields such as `age_group` and `high_risk_count`.

This input format must match the updated `inference.py` used when the endpoint was deployed.

Then use this test code:

```python
import boto3
import json

rt = boto3.client("sagemaker-runtime", region_name=REGION)

# High-risk profile: 55-year-old male with several cardiac risk factors
# This uses the conventional 13 raw input fields.
high_risk = {
    "age": 55,
    "sex": 1,
    "cp": 0,
    "trestbps": 140,
    "chol": 250,
    "fbs": 0,
    "restecg": 0,
    "thalach": 145,
    "exang": 1,
    "oldpeak": 2.3,
    "slope": 1,
    "ca": 1,
    "thal": 7
}

resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(high_risk)
)

result = json.loads(resp["Body"].read())[0]

print("HIGH-RISK PROFILE (55yo male, asymptomatic CP, exercise angina)")
print(f"  Prediction  : {result['label']}")
print(f"  Probability : {result['probability']:.1%}")
```

**Important:** The endpoint expects `ContentType="application/json"`. Using `text/csv` will fail because the deployed `inference.py` for this endpoint expects JSON input.

In [40]:
import boto3
import json

REGION = "ap-southeast-1"

# Change to your own endpoint
ENDPOINT_NAME = "iti113-team40-heart-disease"

rt = boto3.client("sagemaker-runtime", region_name=REGION)


def invoke_heart_endpoint(input_dict):
    """
    Invoke the SageMaker endpoint using the JSON dictionary format
    expected by the updated deployed inference.py.

    The updated endpoint accepts the conventional 13 raw Heart Disease fields.
    The endpoint performs preprocessing internally, including:
    - age_group creation
    - high_risk_count creation
    - numeric scaling
    - feature ordering
    """

    response = rt.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(input_dict)
    )

    result = json.loads(response["Body"].read())[0]
    return result


# Test with high-risk profile
# Conventional 13 raw input fields only
high_risk = {
    "age": 55,
    "sex": 1,
    "cp": 0,
    "trestbps": 140,
    "chol": 250,
    "fbs": 0,
    "restecg": 0,
    "thalach": 145,
    "exang": 1,
    "oldpeak": 2.3,
    "slope": 1,
    "ca": 1,
    "thal": 7
}

low_risk = {
    "age": 42,
    "sex": 0,
    "cp": 1,
    "trestbps": 120,
    "chol": 180,
    "fbs": 0,
    "restecg": 1,
    "thalach": 170,
    "exang": 0,
    "oldpeak": 0.2,
    "slope": 2,
    "ca": 0,
    "thal": 3
}

# Invoke endpoint
try:
    result = invoke_heart_endpoint(high_risk)

    print("\nHIGH-RISK PROFILE")
    print("Prediction :", result["label"])
    print("Probability:", f'{result["probability"]:.1%}')

    result = invoke_heart_endpoint(low_risk)

    print("\nLOW-RISK PROFILE")
    print("Prediction :", result["label"])
    print("Probability:", f'{result["probability"]:.1%}')


except Exception as e:
    print("Endpoint invocation failed.")
    print("Error:", e)


HIGH-RISK PROFILE
Prediction : Heart disease present
Probability: 70.6%

LOW-RISK PROFILE
Prediction : No heart disease
Probability: 5.5%


## 6. Build the Gradio Prediction Function

This function will be connected to the Gradio interface. It collects values from UI fields, sends them to the endpoint, and returns the endpoint response.

In [44]:
import boto3
import json

REGION = "ap-southeast-1"

# change to your own team including ENDPOINT 
TEAM_ID = "team40"
STUDENT_ID = "s4001"
ENDPOINT_NAME = "iti113-team40-heart-disease"

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

In [45]:
def predict_heart_disease(
    age,
    sex,
    cp,
    trestbps,
    chol,
    fbs,
    restecg,
    thalach,
    exang,
    oldpeak,
    slope,
    ca,
    thal
):
    """
    Gradio prediction function.

    Sends conventional 13 raw Heart Disease fields to the SageMaker endpoint.
    The updated endpoint's inference.py performs preprocessing internally:
    - age_group creation
    - high_risk_count creation
    - numeric scaling
    - feature ordering
    """

    input_dict = {
        "age": int(age),
        "sex": int(sex),
        "cp": int(cp),
        "trestbps": int(trestbps),
        "chol": int(chol),
        "fbs": int(fbs),
        "restecg": int(restecg),
        "thalach": int(thalach),
        "exang": int(exang),
        "oldpeak": float(oldpeak),
        "slope": int(slope),
        "ca": int(ca),
        "thal": int(thal)
    }

    try:
        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps(input_dict)
        )

        result = json.loads(response["Body"].read())[0]

        prediction = result.get("label", "Unknown")
        probability = result.get("probability", None)

        if probability is not None:
            probability_text = f"{probability:.1%}"
        else:
            probability_text = "Not returned"

        return (
            f"Endpoint: {ENDPOINT_NAME}\n"
            f"Input JSON: {json.dumps(input_dict, indent=2)}\n\n"
            f"Prediction: {prediction}\n"
            f"Probability: {probability_text}"
        )

    except Exception as e:
        return (
            "Error invoking endpoint.\n"
            f"Endpoint: {ENDPOINT_NAME}\n"
            f"Input JSON: {json.dumps(input_dict, indent=2)}\n\n"
            f"Error: {str(e)}"
        )

## 7. Launch the Gradio UI

Run the cell below to start the front-end UI inside the notebook.

> If the UI does not appear inline, check the printed local URL. In managed Studio environments, external sharing may be disabled, so this notebook uses `share=False` by default.

In [50]:
with gr.Blocks(title="ITI113 Heart Disease Prediction Demo") as demo:
    gr.Markdown(
        f"""
        # ITI113 Heart Disease Prediction Demo

        This simple Gradio UI invokes a SageMaker Serverless Endpoint.

        **Team:** `{TEAM_ID}`  
        **Student/Profile:** `{STUDENT_ID}`  
        **Endpoint:** `{ENDPOINT_NAME}`  
        **Region:** `{REGION}`

        The UI does not train the model. It collects the conventional 13 Heart Disease input fields,
        sends them as a JSON request to the deployed SageMaker endpoint, and displays the prediction result.

        The updated endpoint performs preprocessing internally, including `age_group` creation,
        `high_risk_count` creation, numeric scaling, and feature ordering.
        """
    )

    with gr.Row():
        age = gr.Number(label="Age", value=63)
        sex = gr.Dropdown(label="Sex", choices=[0, 1], value=1, info="0 = female, 1 = male")
        cp = gr.Dropdown(label="Chest Pain Type (cp)", choices=[0, 1, 2, 3], value=3)

    with gr.Row():
        trestbps = gr.Number(label="Resting Blood Pressure", value=160)
        chol = gr.Number(label="Cholesterol", value=286)
        fbs = gr.Dropdown(label="Fasting Blood Sugar > 120", choices=[0, 1], value=1)

    with gr.Row():
        restecg = gr.Dropdown(label="Resting ECG", choices=[0, 1, 2], value=0)
        thalach = gr.Number(label="Maximum Heart Rate", value=108)
        exang = gr.Dropdown(label="Exercise Induced Angina", choices=[0, 1], value=1)

    with gr.Row():
        oldpeak = gr.Number(label="Oldpeak", value=2.8)
        slope = gr.Dropdown(label="Slope", choices=[0, 1, 2], value=0)
        ca = gr.Dropdown(label="Number of Major Vessels (ca)", choices=[0, 1, 2, 3, 4], value=2)
        thal = gr.Dropdown(label="Thal", choices=[0, 1, 2, 3, 6, 7], value=7)

    predict_button = gr.Button("Predict")
    output = gr.Textbox(label="Prediction Result", lines=10)

    predict_button.click(
        fn=predict_heart_disease,
        inputs=[
            age, sex, cp, trestbps, chol, fbs, restecg,
            thalach, exang, oldpeak, slope, ca, thal
        ],
        outputs=output
    )

# In SageMaker Studio, inline=True is usually the most convenient for notebook demos.

# if below fails to show:
# demo.launch(
#     inline=True,
#     share=False,
#     debug=True,
#     server_name="0.0.0.0",
#     server_port=7860
# )
# run this temporarily
demo.launch(
    inline=True,
    share=True,
    debug=True
)

* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://f00a0600453504700d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f00a0600453504700d.gradio.live


In [51]:
demo.close() # to close gradio
# gradio does not close, press stop in jupyter notebook

Closing server running on port: 7860


In [52]:
# run this to confirm gradio closed
import os
import signal
import subprocess

PORT = 7860

cmd = f"lsof -ti:{PORT}"
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

pids = result.stdout.strip().splitlines()

if not pids:
    print(f"No process found on port {PORT}")
else:
    for pid in pids:
        print(f"Killing process {pid} on port {PORT}")
        os.kill(int(pid), signal.SIGTERM)

    print(f"Port {PORT} should now be free.")

No process found on port 7860


## 8. Example Inputs for Students

### Lower-risk style example

| Feature | Value |
|---|---:|
| age | 42 |
| sex | 0 |
| cp | 1 |
| trestbps | 120 |
| chol | 180 |
| fbs | 0 |
| restecg | 1 |
| thalach | 170 |
| exang | 0 |
| oldpeak | 0.2 |
| slope | 2 |
| ca | 0 |
| thal | 2 |

### Higher-risk style example

| Feature | Value |
|---|---:|
| age | 63 |
| sex | 1 |
| cp | 3 |
| trestbps | 160 |
| chol | 286 |
| fbs | 1 |
| restecg | 0 |
| thalach | 108 |
| exang | 1 |
| oldpeak | 2.8 |
| slope | 0 |
| ca | 2 |
| thal | 3 |

## 9. Troubleshooting

| Problem | Possible Cause | What to Check |
|---|---|---|
| `ValidationException: Could not find endpoint` | Wrong endpoint name | Run `list_endpoints()` and copy the exact endpoint name. |
| `AccessDeniedException` | Wrong team role or endpoint outside team permission | Check that you are using your own SageMaker Studio profile and own team endpoint. |
| `ModelError` | Payload format does not match `inference.py` | Check whether your endpoint expects CSV or JSON. |
| Gradio UI does not appear | Studio/browser rendering issue | Check the printed local URL, or restart the notebook kernel and rerun. |
| Endpoint charges continue | Endpoint still exists | Delete the endpoint after testing if it is no longer needed. |

For this tutorial, students should change:

```python
TEAM_ID = "teamXX"
STUDENT_ID = "sXXXX"
ENDPOINT_NAME = "your-team-endpoint-name"
```

## 10. Optional Cleanup Reminder

Do **not** run cleanup if the endpoint is still needed for demonstration.

When finished, delete the endpoint from the SageMaker console or using `boto3` to avoid leaving unused resources active.

In [ ]:
# Optional cleanup example. Uncomment only when you really want to delete the endpoint.

# endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
# endpoint_config_name = endpoint_desc["EndpointConfigName"]

# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
# print("Deleted endpoint:", ENDPOINT_NAME)

# sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
# print("Deleted endpoint config:", endpoint_config_name)